# 03. Graph-Feature Baselines

This notebook sits between plain tabular modeling and full GNNs.

It computes simple structural features from the sampled edge list and fuses them with the compact wallet feature block.


In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
nodes_df = pd.read_csv(ROOT / "wash_trading_gnn_nodes_10000.csv")
edges_df = pd.read_csv(ROOT / "wash_trading_gnn_edges_10000.csv")


In [ ]:
G = nx.from_pandas_edgelist(
    edges_df,
    source="src_node_id",
    target="dst_node_id",
    create_using=nx.DiGraph(),
)

pagerank = nx.pagerank(G, alpha=0.85)
weak_component_size = {}
for component in nx.weakly_connected_components(G):
    size = len(component)
    for node in component:
        weak_component_size[node] = size

graph_features = pd.DataFrame(
    {
        "node_id": nodes_df["node_id"],
        "nx_in_degree": nodes_df["node_id"].map(dict(G.in_degree())).fillna(0).astype(float),
        "nx_out_degree": nodes_df["node_id"].map(dict(G.out_degree())).fillna(0).astype(float),
        "nx_total_degree": nodes_df["node_id"].map(dict(G.degree())).fillna(0).astype(float),
        "nx_pagerank": nodes_df["node_id"].map(pagerank).fillna(0.0),
        "nx_component_size": nodes_df["node_id"].map(weak_component_size).fillna(1).astype(float),
        "nx_has_self_loop": nodes_df["node_id"].isin(nx.nodes_with_selfloops(G)).astype(int),
    }
)

display(graph_features.head())


In [ ]:
fused_df = nodes_df.merge(graph_features, on="node_id", how="left")

base_features = [c for c in fused_df.columns if c.startswith("eth_twitter_combined_features_")]
graph_stats = [
    "full_in_degree",
    "full_out_degree",
    "full_total_degree",
    "full_positive_touch_count",
    "full_has_self_loop",
    "sub_in_degree",
    "sub_out_degree",
    "sub_total_degree",
    "nx_in_degree",
    "nx_out_degree",
    "nx_total_degree",
    "nx_pagerank",
    "nx_component_size",
    "nx_has_self_loop",
]

X_base = fused_df[base_features].copy()
X_fused = fused_df[base_features + graph_stats].copy()
y = fused_df["label"].copy()


In [ ]:
def best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_t, best_f1 = 0.5, -1.0
    for threshold in thresholds:
        score = f1_score(y_true, (y_prob >= threshold).astype(int), zero_division=0)
        if score > best_f1:
            best_t, best_f1 = float(threshold), float(score)
    return best_t


def evaluate(name, y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "model": name,
        "threshold": threshold,
        "PR-AUC": average_precision_score(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Balanced-Accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
    }


In [ ]:
RANDOM_STATE = 42
train_idx, temp_idx = train_test_split(
    fused_df.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y.loc[temp_idx],
)

y_train, y_val, y_test = y.loc[train_idx], y.loc[val_idx], y.loc[test_idx]
scale_pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()


In [ ]:
def run_xgb(X_train, X_val, X_test):
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_lambda=1.0,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    scale_pos_weight=scale_pos_weight,
                    tree_method="hist",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]
    threshold = best_threshold(y_val.to_numpy(), val_prob)
    return evaluate("xgboost", y_test.to_numpy(), test_prob, threshold)


def run_logreg(X_train, X_val, X_test):
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    )
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]
    threshold = best_threshold(y_val.to_numpy(), val_prob)
    return evaluate("logreg", y_test.to_numpy(), test_prob, threshold)


In [ ]:
comparison_rows = []
for dataset_name, X_data in [("node_features_only", X_base), ("node_plus_graph_stats", X_fused)]:
    comparison_rows.append({"dataset": dataset_name, **run_logreg(X_data.loc[train_idx], X_data.loc[val_idx], X_data.loc[test_idx])})
    comparison_rows.append({"dataset": dataset_name, **run_xgb(X_data.loc[train_idx], X_data.loc[val_idx], X_data.loc[test_idx])})

comparison_df = pd.DataFrame(comparison_rows).sort_values(["PR-AUC", "F1"], ascending=False)
display(comparison_df)


## Interpretation

If `node_plus_graph_stats` clearly beats `node_features_only`, then graph structure is already helpful even before a GNN.
That makes the GNN comparison meaningful instead of decorative.
